# Convolutional Neural Networks (CNNs)
**SoC 2026 | Week 4**

**Topics (StatQuest 86–91):**
- Convolution operation
- Feature maps & filters
- Pooling layers
- CNN architecture overview
- Why CNNs work for images (parameter sharing, translation invariance)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d

## 1. The Convolution Operation

A **filter (kernel)** slides over the image, computing dot products at each position to produce a **feature map**.

Key intuition: the same filter detects the same feature anywhere in the image — this is **parameter sharing**.

In [ ]:
# Synthetic grayscale 'image'
np.random.seed(7)
image = np.zeros((28, 28))
image[5:23, 5:23] = 1      # Square
image[10:18, 10:18] = 0    # Hole in square
image += np.random.randn(28, 28) * 0.1   # Add noise

# Edge detection kernels
horizontal_edge = np.array([[-1,-1,-1],[0,0,0],[1,1,1]])
vertical_edge   = np.array([[-1,0,1],[-1,0,1],[-1,0,1]])
blur            = np.ones((3,3)) / 9

filters = [horizontal_edge, vertical_edge, blur]
names   = ['Horizontal Edge', 'Vertical Edge', 'Blur (Avg Pool)']

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Original')

for ax, f, name in zip(axes[1:], filters, names):
    feature_map = convolve2d(image, f, mode='same')
    ax.imshow(feature_map, cmap='RdBu')
    ax.set_title(name)

for ax in axes: ax.axis('off')
plt.suptitle('Convolution with Different Kernels', fontsize=12)
plt.tight_layout(); plt.show()

## 2. Pooling

Pooling **downsamples** feature maps, reducing computation and providing translation invariance.

- **Max pooling**: takes the maximum value in each window (most common)
- **Average pooling**: takes the mean

In [ ]:
def max_pool2d(x, pool_size=2, stride=2):
    h, w = x.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    out   = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            out[i, j] = x[i*stride:i*stride+pool_size,
                          j*stride:j*stride+pool_size].max()
    return out

# Apply edge filter then pool
edge_map  = convolve2d(image, vertical_edge, mode='same')
pooled    = max_pool2d(edge_map, pool_size=2, stride=2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(edge_map,  cmap='RdBu'); axes[0].set_title(f'Feature Map {edge_map.shape}')
axes[1].imshow(pooled,    cmap='RdBu'); axes[1].set_title(f'After Max Pool {pooled.shape}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Typical CNN Architecture

```
Input Image
   │
   ├─ Conv(32 filters, 3×3) + ReLU
   ├─ MaxPool(2×2)
   │
   ├─ Conv(64 filters, 3×3) + ReLU
   ├─ MaxPool(2×2)
   │
   ├─ Flatten
   ├─ Dense(128) + ReLU
   ├─ Dropout(0.5)
   └─ Dense(n_classes) + Softmax
         │
      Prediction
```

**Why CNNs beat fully-connected nets on images:**
1. **Parameter sharing** — one filter detects the same feature everywhere
2. **Local connectivity** — each neuron only looks at a small patch
3. **Translation invariance** — pooling makes features location-agnostic
4. **Hierarchical features** — early layers: edges → mid: shapes → deep: objects

## 4. Coming Up — PyTorch CNN (Week 5)

In Week 5 we'll implement a CNN in PyTorch using `nn.Conv2d`, `nn.MaxPool2d`, and train it on MNIST.

See `../week5_pytorch/pytorch_intro.ipynb`.